# Subsecuencia creciente más larga (LIS)

Buscar la subsecuencia estrictamente creciente más larga de una secuencia. $L[i]$ es la longitud de la más larga que termina en $a_i$: $L[i] = 1 + \max\{L[j] : j < i,\ a_j < a_i\}$ (o 1 si no hay ninguna), con coste $O(n^2)$. Guardando para cada longitud el menor final posible y buscando con `bisect` baja a $O(n \log n)$.

## 1. Versión $O(n^2)$ con reconstrucción

Calcular $L[i]$ guardando también de qué posición viene cada una, y reconstruir la subsecuencia desde la posición con mayor $L$.

In [1]:
def lis_cuadratica(a):
    if not a:
        return []
    L, previo = [1] * len(a), [None] * len(a)
    for i in range(len(a)):
        for j in range(i):
            if a[j] < a[i] and L[j] + 1 > L[i]:
                L[i], previo[i] = L[j] + 1, j
    i = max(range(len(a)), key=lambda k: L[k])
    resultado = []
    while i is not None:
        resultado.append(a[i])
        i = previo[i]
    return resultado[::-1]

print(lis_cuadratica([10, 9, 2, 5, 3, 7, 101, 18]))
print(lis_cuadratica([3, 1, 4, 1, 5, 9, 2, 6, 5, 3, 5, 8, 9, 7, 9]))

[2, 5, 7, 101]
[3, 4, 5, 6, 8, 9]


## 2. Versión $O(n \log n)$ con `bisect`

`finales[k]` es el menor valor en que termina una subsecuencia creciente de longitud $k + 1$. La lista está ordenada, así que cada elemento se coloca con `bisect_left`: si va al final alarga la mejor subsecuencia y, si no, mejora un final. Guardando el índice de cada final y el predecesor de cada elemento se reconstruye la subsecuencia. Comparar tiempos con la versión cuadrática.

In [2]:
import random
import time
from bisect import bisect_left

def lis_rapida(a):
    finales, indices, previo = [], [], [None] * len(a)
    for i, x in enumerate(a):
        k = bisect_left(finales, x)
        if k == len(finales):
            finales.append(x)
            indices.append(i)
        else:
            finales[k], indices[k] = x, i
        previo[i] = indices[k - 1] if k else None
    resultado, i = [], indices[-1] if indices else None
    while i is not None:
        resultado.append(a[i])
        i = previo[i]
    return resultado[::-1]

print(lis_rapida([10, 9, 2, 5, 3, 7, 101, 18]))
print(lis_rapida([3, 1, 4, 1, 5, 9, 2, 6, 5, 3, 5, 8, 9, 7, 9]))

random.seed(9)
datos = [random.randint(0, 10**6) for _ in range(5000)]
for f in (lis_cuadratica, lis_rapida):
    inicio = time.perf_counter()
    resultado = f(datos)
    print(f"{f.__name__}: longitud {len(resultado)} en {(time.perf_counter() - inicio) * 1000:.1f} ms")

[2, 3, 7, 18]
[1, 2, 3, 5, 7, 9]


lis_cuadratica: longitud 140 en 402.5 ms
lis_rapida: longitud 140 en 0.6 ms


## 3. Comprobación con fuerza bruta

Con listas aleatorias pequeñas, probar las subsecuencias de mayor a menor longitud hasta encontrar una creciente, y comprobar que las dos versiones dan la misma longitud y una subsecuencia creciente de verdad.

In [3]:
from itertools import combinations

def es_creciente(s):
    return all(x < y for x, y in zip(s, s[1:]))

def es_subsecuencia(s, t):
    resto = iter(t)
    return all(x in resto for x in s)

def lis_fuerza_bruta(a):
    return next(r for r in range(len(a), -1, -1) if any(es_creciente(c) for c in combinations(a, r)))

random.seed(10)
for _ in range(500):
    a = [random.randint(0, 9) for _ in range(random.randint(0, 11))]
    optimo = lis_fuerza_bruta(a)
    for f in (lis_cuadratica, lis_rapida):
        s = f(a)
        assert len(s) == optimo and es_creciente(s) and es_subsecuencia(s, a)

## 4. Subsecuencia no decreciente

Para admitir valores repetidos (subsecuencia no decreciente) basta con cambiar `bisect_left` por `bisect_right`: un valor igual al final de una subsecuencia la alarga en lugar de sustituirlo. Comprobarlo con fuerza bruta.

In [4]:
from bisect import bisect_right

def lnds(a):
    finales = []
    for x in a:
        k = bisect_right(finales, x)
        if k == len(finales):
            finales.append(x)
        else:
            finales[k] = x
    return len(finales)

def es_no_decreciente(s):
    return all(x <= y for x, y in zip(s, s[1:]))

print(lnds([3, 1, 4, 1, 5, 9, 2, 6, 5, 3, 5, 8, 9, 7, 9]), len(lis_rapida([2, 2, 2, 2])), lnds([2, 2, 2, 2]))

random.seed(11)
for _ in range(500):
    a = [random.randint(0, 5) for _ in range(random.randint(0, 11))]
    assert lnds(a) == next(r for r in range(len(a), -1, -1) if any(es_no_decreciente(c) for c in combinations(a, r)))

8 1 4


## 5. Cajas que caben unas dentro de otras

Una caja de $a \times b$ cabe dentro de otra de $c \times d$ si $a < c$ y $b < d$ (sin girarlas). ¿Cuántas cajas se pueden meter unas dentro de otras como máximo? Ordenando por anchura creciente y, a igual anchura, por altura decreciente (para no meter dos de la misma anchura), la respuesta es la LIS de las alturas.

In [5]:
def max_anidadas(cajas):
    ordenadas = sorted(cajas, key=lambda c: (c[0], -c[1]))
    return len(lis_rapida([alto for _, alto in ordenadas]))

def anidadas_fuerza_bruta(cajas):
    return max(r for r in range(len(cajas) + 1) for c in combinations(sorted(cajas), r)
               if all(x[0] < y[0] and x[1] < y[1] for x, y in zip(c, c[1:])))

cajas = [(5, 4), (6, 4), (6, 7), (2, 3), (1, 1), (8, 9), (7, 8), (3, 9)]
print(max_anidadas(cajas))

random.seed(12)
for _ in range(300):
    cajas = [(random.randint(1, 6), random.randint(1, 6)) for _ in range(random.randint(0, 10))]
    assert max_anidadas(cajas) == anidadas_fuerza_bruta(cajas)

6
